In [0]:
# =============================================================================
# Silver Layer ETL — Cybersecurity Threat Detection
# Source : workspace.default.cybersecurity_training_bronze
# Target : workspace.default.cybersecurity_training_silver
# =============================================================================

from pyspark.sql.functions import col, trim, when
from pyspark.sql.types import StringType, NumericType

BRONZE_TABLE = "workspace.default.cybersecurity_training_bronze"
SILVER_TABLE = "workspace.default.cybersecurity_training_silver"

# ---------------------------------------------------------------------------
# 1. Load the Bronze Delta table
# ---------------------------------------------------------------------------
bronze_df = spark.table(BRONZE_TABLE)

# ---------------------------------------------------------------------------
# 2. Print the Bronze row count and schema
# ---------------------------------------------------------------------------
print("=== Bronze schema (raw) ===")
bronze_df.printSchema()

bronze_count = bronze_df.count()
print("Bronze row count: " + str(bronze_count))
print("")

print("Original columns: " + str(bronze_df.columns))
print("")

# ---------------------------------------------------------------------------
# 3. Remove duplicate rows
# ---------------------------------------------------------------------------
silver_df = bronze_df.dropDuplicates()

after_dedup = silver_df.count()
removed = bronze_count - after_dedup
print("Rows after deduplication: " + str(after_dedup) + "  (removed " + str(removed) + ")")
print("")

# ---------------------------------------------------------------------------
# 4. Inspect the actual schema before transforming columns
# ---------------------------------------------------------------------------
cleaned_columns = silver_df.columns

string_cols = [f.name for f in silver_df.schema.fields if isinstance(f.dataType, StringType)]
numeric_cols = [f.name for f in silver_df.schema.fields if isinstance(f.dataType, NumericType)]

print("String columns: " + str(string_cols))
print("Numeric columns: " + str(numeric_cols))
print("")

# ---------------------------------------------------------------------------
# 5. Trim whitespace from string columns and convert empty strings to NULL
# ---------------------------------------------------------------------------
for c in string_cols:
    silver_df = silver_df.withColumn(
        c,
        when(trim(col(c)) == "", None).otherwise(trim(col(c))),
    )

print("Trimmed whitespace and converted empty strings to NULL in " + str(len(string_cols)) + " string columns.")
print("")

# ---------------------------------------------------------------------------
# 6. Safely convert numeric-like columns to DOUBLE
#    Only convert columns that are currently StringType and whose name hints
#    at network-traffic statistics.  Do not touch already-numeric columns
#    and do not touch categorical columns.
# ---------------------------------------------------------------------------
traffic_hints = [
    "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss",
    "spkts", "dpkts", "sload", "dload", "smean", "dmean",
    "sinpkt", "dinpkt", "sjit", "djit", "swin", "dwin",
    "stcpb", "dtcpb", "tcprtt", "synack", "ackdat",
    "sintpkt", "dintpkt", "rate", "srate", "drate",
    "bytes", "pkts", "load", "mean", "ttl", "loss", "win",
    "jit", "inpkt", "intpkt", "tcpb", "rtt", "ack", "syn",
]

categorical_hints = ["proto", "service", "state", "attack_cat", "category", "label"]

numeric_converted = []
for c in cleaned_columns:
    if c in string_cols:
        c_lower = c.lower()
        is_categorical = False
        for cat in categorical_hints:
            if cat in c_lower:
                is_categorical = True
                break
        if is_categorical:
            continue
        is_traffic = False
        for hint in traffic_hints:
            if hint in c_lower:
                is_traffic = True
                break
        if is_traffic:
            silver_df = silver_df.withColumn(c, col(c).cast("double"))
            numeric_converted.append(c)

print("Converted " + str(len(numeric_converted)) + " columns to double:")
for c in numeric_converted:
    print("  " + c)
print("")

# ---------------------------------------------------------------------------
# 7. Keep categorical columns (proto, service, state) as STRING
# ---------------------------------------------------------------------------
for cat_col in ["proto", "service", "state"]:
    if cat_col in silver_df.columns:
        silver_df = silver_df.withColumn(cat_col, col(cat_col).cast("string"))

print("Categorical columns proto, service, state confirmed as string.")
print("")

# ---------------------------------------------------------------------------
# 8. Convert the label column to INTEGER if it exists
# ---------------------------------------------------------------------------
if "label" in silver_df.columns:
    silver_df = silver_df.withColumn("label", col("label").cast("int"))
    print("Converted 'label' to integer.")
else:
    print("WARNING: 'label' column not found in the Bronze table.")
print("")

# ---------------------------------------------------------------------------
# 9. Remove rows where label is NULL
# ---------------------------------------------------------------------------
if "label" in silver_df.columns:
    before_null_drop = silver_df.count()
    silver_df = silver_df.filter(col("label").isNotNull())
    after_null_drop = silver_df.count()
    null_removed = before_null_drop - after_null_drop
    print("Removed " + str(null_removed) + " rows with NULL label. Remaining: " + str(after_null_drop))
    print("")

# ---------------------------------------------------------------------------
# 10. All columns preserved — no columns dropped
# ---------------------------------------------------------------------------
print("All cybersecurity-relevant columns preserved (no columns dropped).")
print("")

# ---------------------------------------------------------------------------
# 11. Display the cleaned Silver data
# ---------------------------------------------------------------------------
print("=" * 60)
print("SILVER LAYER — FINAL")
print("=" * 60)

print("")
print("--- Final Schema ---")
silver_df.printSchema()

final_count = silver_df.count()
print("")
print("Final row count: " + str(final_count))

print("")
print("--- Sample Records ---")
display(silver_df.limit(20))

# ---------------------------------------------------------------------------
# 12. Save the result as a Delta table
# ---------------------------------------------------------------------------
(silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print("")
print("Saved Silver table to: " + SILVER_TABLE)

# ---------------------------------------------------------------------------
# 13. Final verification — read back and show row count and schema
# ---------------------------------------------------------------------------
silver_check = spark.table(SILVER_TABLE)
check_count = silver_check.count()

print("")
print("=" * 60)
print("VERIFICATION — reading back from Silver table")
print("=" * 60)
print("Row count in saved Silver table: " + str(check_count))
print("")
print("--- Saved Silver Schema ---")
silver_check.printSchema()

print("")
print("Silver layer created successfully")

=== Bronze schema (raw) ===
root
 |-- id: integer (nullable = true)
 |-- dur: double (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- state: string (nullable = true)
 |-- spkts: integer (nullable = true)
 |-- dpkts: integer (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- rate: double (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sload: double (nullable = true)
 |-- dload: double (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- sinpkt: double (nullable = true)
 |-- dinpkt: double (nullable = true)
 |-- sjit: double (nullable = true)
 |-- djit: double (nullable = true)
 |-- swin: integer (nullable = true)
 |-- stcpb: long (nullable = true)
 |-- dtcpb: long (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- tcprtt: double (nullable = true)
 |-- synack: double (nullable = true)
 |

id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
1,0.121478,tcp,-,FIN,6,4,258,172,74.08749,252,254,14158.94238,8495.365234,0,0,24.2956,8.375,30.177547,11.830604,255,621772692,2202533631,255,0.0,0.0,0.0,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0
2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,62,252,8395.112305,503571.3125,2,17,49.915,15.432865,61.426934,1387.77833,255,1417884146,3077387971,255,0.0,0.0,0.0,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0
3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,62,252,1572.271851,60929.23047,1,6,231.875571,102.737203,17179.58686,11420.92623,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0
4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,62,252,2740.178955,3358.62207,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.0,0.0,0.0,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0
5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0
6,0.380537,tcp,-,FIN,10,6,534,268,39.41798,254,252,10112.02539,4709.134766,2,1,39.928778,52.241,2223.730342,82.5505,255,3984155503,1796040391,255,0.172934,0.119331,0.053603,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0
7,0.637109,tcp,-,FIN,10,8,534,354,26.683033,254,252,6039.783203,3892.58374,2,1,68.267778,81.137711,4286.82857,119.422719,255,1787309226,1767180493,255,0.143337,0.069136,0.074201,53,44,0,0,43,1,1,1,1,40,0,0,0,1,39,0,Normal,0
8,0.521584,tcp,-,FIN,10,8,534,354,32.593026,254,252,7377.527344,4754.74707,2,1,55.794,66.054141,3770.580726,118.962633,255,205985702,316006300,255,0.116615,0.059195,0.05742,53,44,0,0,43,1,3,3,1,40,0,0,0,3,39,0,Normal,0
9,0.542905,tcp,-,FIN,10,8,534,354,31.313031,254,252,7087.796387,4568.018555,2,1,60.210889,68.109,4060.625597,106.611547,255,884094874,3410317203,255,0.118584,0.066133,0.052451,53,44,0,0,43,1,3,3,1,40,0,0,0,3,39,0,Normal,0
10,0.258687,tcp,-,FIN,10,6,534,268,57.985135,254,252,14875.12012,6927.291016,2,1,27.505111,39.106801,1413.686415,57.200395,255,3368447996,584859215,255,0.087934,0.063116,0.024818,53,45,0,0,43,1,3,3,1,40,0,0,0,3,39,0,Normal,0



Saved Silver table to: workspace.default.cybersecurity_training_silver

VERIFICATION — reading back from Silver table
Row count in saved Silver table: 175341

--- Saved Silver Schema ---
root
 |-- id: integer (nullable = true)
 |-- dur: double (nullable = true)
 |-- proto: string (nullable = true)
 |-- service: string (nullable = true)
 |-- state: string (nullable = true)
 |-- spkts: integer (nullable = true)
 |-- dpkts: integer (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- rate: double (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sload: double (nullable = true)
 |-- dload: double (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- sinpkt: double (nullable = true)
 |-- dinpkt: double (nullable = true)
 |-- sjit: double (nullable = true)
 |-- djit: double (nullable = true)
 |-- swin: integer (nullable = true)
 |-- stcpb: long (nullabl